# CySent Live RL Training — Phi-3.5-mini-instruct

Real-time REINFORCE + LoRA training against the live CySent environment.
**Not** static dataset — the model observes, acts, and learns from live env rewards.

| Setting | Value |
|---------|-------|
| Model | `microsoft/Phi-3.5-mini-instruct` (3.8B, fp16) |
| RL algo | REINFORCE with baseline |
| LoRA r | 8 (auto-detects `qkv_proj` for Phi arch) |
| Episodes | 200 (max) |
| Max steps/ep | 40 |
| GPU | Colab T4 (free tier sufficient) |
| Est. time | 25-40 min |
| Est. cost | Free tier / ~$0.50 if paid |

In [2]:
!cd /content/CySent && git pull
!rm -rf ~/.cache/huggingface/modules/transformers_modules/microsoft/

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 552 bytes | 110.00 KiB/s, done.
From https://github.com/sxchin-01/CySent
   3a5895e..0475415  main       -> origin/main
Updating 3a5895e..0475415
Fast-forward
 notebooks/CySent_Phi_LiveRL.ipynb | 46 +++++++++++++++++++--------------------
 1 file changed, 23 insertions(+), 23 deletions(-)


In [2]:
# Cell 1 — Install dependencies
!pip install -q --upgrade pip
!pip install -q "transformers>=4.44.2" "accelerate>=0.31.0" peft bitsandbytes
!pip install -q stable-baselines3 gymnasium numpy

# Clone CySent repo
import os
if not os.path.exists('/content/CySent'):
    !git clone https://github.com/sxchin-01/CySent.git /content/CySent

os.chdir('/content/CySent')
!pip install -q -r backend/requirements.txt
os.environ['PYTHONPATH'] = '/content/CySent'
print('Setup complete. If versions changed significantly, Runtime > Restart runtime once.')

Setup complete. If versions changed significantly, Runtime > Restart runtime once.


In [3]:
# Cell 2 — Live RL Training (Phi-3.5)
import sys, os
os.chdir('/content/CySent')
sys.path.insert(0, '/content/CySent')

from backend.train.train_hf_rl import train_hf_rl

summary = train_hf_rl(
    model_id='microsoft/Phi-3.5-mini-instruct',
    output_dir='outputs/phi35_rl_adapter',
    episodes=200,
    max_steps=40,
    lr=5e-5,
    gamma=0.97,
    temperature=0.7,
    lora_r=8,
    lora_alpha=16,
    seed=42,
    grad_accum=4,
    early_stop_reward=3.0,
    save_every=25,
)

import json
print('\n' + json.dumps(summary, indent=2))

[train_hf_rl] Device: cuda
[train_hf_rl] Model: microsoft/Phi-3.5-mini-instruct
[train_hf_rl] Episodes: 200, max_steps: 40


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

[train_hf_rl] LoRA targets: ['qkv_proj', 'o_proj']
[train_hf_rl] Trainable: 4,718,592 / 3,825,798,144 (0.12%)
  ep   0 | reward  -24.22 | risk 0.297 | breaches 2 | steps  37 | loss -77.1573 | bank/credential_thief
  ep   1 | reward  +28.65 | risk 0.257 | breaches 2 | steps  40 | loss 50.3201 | saas/ransomware_gang
  ep   2 | reward  +12.20 | risk 0.116 | breaches 0 | steps  40 | loss 22.9497 | bank/credential_thief
  ep   3 | reward  -10.36 | risk 0.138 | breaches 1 | steps  40 | loss -34.6052 | bank/credential_thief
  ep   4 | reward  +10.12 | risk 0.190 | breaches 2 | steps  40 | loss 23.6687 | bank/ransomware_gang
  ep   5 | reward  -15.38 | risk 0.146 | breaches 1 | steps  40 | loss -46.6757 | saas/credential_thief
  ep   6 | reward  +33.77 | risk 0.174 | breaches 2 | steps  40 | loss 34.0493 | saas/credential_thief
  ep   7 | reward  +84.35 | risk 0.180 | breaches 1 | steps  40 | loss 113.9248 | saas/credential_thief
[train_hf_rl] Early stop: avg reward 20.50 >= 3.0

[train_hf_rl]

In [4]:
# Cell 3 — 4-way Benchmark: phi35_rl vs qwen_rl vs PPO vs random
from backend.train.train_hf_rl import benchmark_hf_adapter
import json

results = benchmark_hf_adapter(
    model_id='microsoft/Phi-3.5-mini-instruct',
    adapter_path='outputs/phi35_rl_adapter/final_adapter',
    episodes=10,
    max_steps=40,
    seed=42,
    extra_adapters=[
        ('qwen_rl', 'Qwen/Qwen2.5-3B-Instruct', 'outputs/hf_rl_adapter/final_adapter'),
    ],
)

print('\n=== Benchmark Results ===')
header = f"{'Agent':<24} {'Avg Reward':>12} {'Avg Risk':>10} {'Avg Breaches':>14} {'Avg Steps':>10}"
print(header)
print('-' * len(header))
for agent, stats in results.items():
    print(f"{agent:<24} {stats['avg_reward']:>+12.3f} {stats['avg_risk']:>10.3f} "
          f"{stats['avg_breaches']:>14.3f} {stats['avg_steps']:>10.1f}")

from pathlib import Path
out = Path('outputs/phi35_rl_adapter/benchmark_results.json')
out.write_text(json.dumps(results, indent=2))
print(f'\nSaved to {out}')

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



=== Benchmark Results ===
Agent                      Avg Reward   Avg Risk   Avg Breaches  Avg Steps
--------------------------------------------------------------------------
hf_rl_adapter                 -17.324      0.257          1.200       38.2
random                        +22.370      0.117          0.300       40.0

Saved to outputs/phi35_rl_adapter/benchmark_results.json


In [5]:
# Cell 4 — Save adapter to Google Drive (optional)
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
dst = '/content/drive/MyDrive/cysent_phi35_rl_adapter'
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree('outputs/phi35_rl_adapter/final_adapter', dst)
print(f'Adapter copied to {dst}')

shutil.copy2('outputs/phi35_rl_adapter/benchmark_results.json',
             '/content/drive/MyDrive/cysent_phi35_benchmark.json')
print('Benchmark results copied to Drive.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Adapter copied to /content/drive/MyDrive/cysent_phi35_rl_adapter
Benchmark results copied to Drive.
